# Cleaning 02: Text, Geography, and Feature-Safety

Use this notebook to normalize city/state text, standardize Risk, remove leakage, and build feature-safety flags.

In [9]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from helpers.logger import log_step, save_log
from helpers.data_loader import DataLoader
from helpers.profile import profile

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

STAGE1_PATH = DataLoader.interim("cleaned_nulls_dups.csv")
STAGE2_PATH = DataLoader.interim("cleaning/cleaning_stage2.csv")
STAGE2_LOG_PATH = DataLoader.interim("cleaning/cleaning_stage2_log.csv")

# print(STAGE1_PATH)

df_clean = pd.read_csv(STAGE1_PATH)
print('Loaded stage 1 shape:', df_clean.shape)

Loaded stage 1 shape: (196825, 55)


In [10]:
profile(df_clean, 'Stage 1 input profile').head(20)

=== Stage 1 input profile ===
shape: (196825, 55)
full duplicates: 0
duplicate Inspection ID: 149


,missing_count,missing_pct,dtype
APPLICATION CREATED DATE,178239,90.56,str
LICENSE STATUS CHANGE DATE,155935,79.23,str
SSA,134988,68.58,float64
Violations,52266,26.55,str
BUSINESS ACTIVITY,20600,10.47,str
BUSINESS ACTIVITY ID,20600,10.47,str
NEIGHBORHOOD,11796,5.99,str
COMMUNITY AREA,11567,5.88,float64
COMMUNITY AREA NAME,11567,5.88,str
BL_LATITUDE,11180,5.68,float64


In [11]:
# Explore 1: City and State anomalies before normalization
city_values = df_clean['City'].dropna().astype(str)
state_values = df_clean['State'].dropna().astype(str)

print('Unique CITY values after upper-strip:', city_values.str.upper().str.strip().nunique())
print('Top 20 CITY values (raw):')
print(city_values.value_counts().head(20))

unexpected_states = sorted(set(state_values.str.upper().str.strip()) - {'IL'})
print('\nUnexpected STATE values:', unexpected_states)
print('Count rows with State != IL after normalization logic:', int((state_values.str.upper().str.strip() != 'IL').sum()))

Unique CITY values after upper-strip: 66
Top 20 CITY values (raw):
City
CHICAGO              195986
Chicago                 320
chicago                  97
CCHICAGO                 46
SCHAUMBURG               25
MAYWOOD                  15
ELK GROVE VILLAGE        13
CHicago                  12
EVANSTON                 11
CHESTNUT STREET          10
CICERO                    9
SKOKIE                    8
INACTIVE                  8
NILES NILES               7
CHICAGOCHICAGO            7
ELMHURST                  6
CALUMET CITY              6
CHCHICAGO                 6
OAK PARK                  5
WORTH                     5
Name: count, dtype: int64

Unexpected STATE values: ['IN', 'NY', 'WI']
Count rows with State != IL after normalization logic: 3


In [12]:
# Action 1: Normalize key text fields, add city/state flags, then drop near-constant columns
rows_before, cols_before = df_clean.shape
for col in ['City', 'State', 'DBA Name', 'AKA Name', 'Address']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('string').str.strip().str.upper()

if 'City' in df_clean.columns:
    city_norm = df_clean['City'].str.upper().str.replace(r'\s+', ' ', regex=True)
    city_norm = city_norm.str.replace(r'^C+HICAGO$', 'CHICAGO', regex=True)
    df_clean['City'] = city_norm

if 'State' in df_clean.columns:
    df_clean['State'] = df_clean['State'].str.upper()

if 'State' in df_clean.columns:
    df_clean['flag_non_il_state'] = df_clean['State'].notna() & (df_clean['State'] != 'IL')
if 'City' in df_clean.columns:
    df_clean['flag_non_chicago_city'] = df_clean['City'].notna() & (df_clean['City'] != 'CHICAGO')

drop_geo_cols = [c for c in ['City', 'State'] if c in df_clean.columns]
if drop_geo_cols:
    df_clean = df_clean.drop(columns=drop_geo_cols)

log_step('normalize_text_fields_and_flags', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Normalize text, add flags, drop City/State')
print('Shape after city/state normalization:', df_clean.shape)

Shape after city/state normalization: (196825, 55)


In [13]:
# Action 2: Add longitude quality flag (preserve rows)
rows_before, cols_before = df_clean.shape
longitude = pd.to_numeric(df_clean['Longitude'], errors='coerce') if 'Longitude' in df_clean.columns else pd.Series(dtype='float64')

if 'Longitude' in df_clean.columns:
    df_clean['flag_longitude_outside_typical_range'] = (
        df_clean['Longitude'].notna() & ((df_clean['Longitude'] < -87.95) | (df_clean['Longitude'] > -87.5))
    )

log_step('add_longitude_flag', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Flag out-of-range longitude values')
print('Longitude quantiles:')
print(longitude.quantile([0.0, 0.01, 0.05, 0.5, 0.95, 0.99, 1.0]))
print('Flagged longitude anomalies:', int(df_clean['flag_longitude_outside_typical_range'].sum()) if 'flag_longitude_outside_typical_range' in df_clean.columns else 'N/A')

Longitude quantiles:
0.00   -87.914428
0.01   -87.914428
0.05   -87.778516
0.50   -87.666812
0.95   -87.601401
0.99   -87.552631
1.00   -87.525094
Name: Longitude, dtype: float64
Flagged longitude anomalies: 0


In [14]:
# Action 3: Cast types, remove leakage, and build informative flags
rows_before, cols_before = df_clean.shape

int_like_cols = [
    'Inspection ID', 'License #', 'LICENSE NUMBER', 'BL_LICENSE_ID', 'ACCOUNT NUMBER',
    'SITE NUMBER', 'WARD', 'PRECINCT', 'POLICE DISTRICT', 'COMMUNITY AREA',
    'LICENSE CODE', 'SSA', 'BL_ZIP_CODE'
]
for col in int_like_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').round().astype('Int64')

float_cols = ['Latitude', 'Longitude', 'BL_LATITUDE', 'BL_LONGITUDE']
for col in float_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

date_cols = [
    'Inspection Date', 'APPLICATION CREATED DATE', 'APPLICATION REQUIREMENTS COMPLETE',
    'PAYMENT DATE', 'LICENSE TERM START DATE', 'LICENSE TERM EXPIRATION DATE',
    'LICENSE APPROVED FOR ISSUANCE', 'DATE ISSUED', 'LICENSE STATUS CHANGE DATE'
]
for col in date_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

if 'Zip' in df_clean.columns:
    zip_num = pd.to_numeric(df_clean['Zip'], errors='coerce')
    df_clean['Zip'] = zip_num.astype('Int64').astype('string').str.zfill(5)
    df_clean.loc[zip_num.isna(), 'Zip'] = pd.NA

if 'Risk' in df_clean.columns:
    risk_raw = df_clean['Risk'].astype('string').str.strip()
    risk_map = {
        'Risk 1 (High)': 'High',
        'Risk 2 (Medium)': 'Medium',
        'Risk 3 (Low)': 'Low',
        'All': pd.NA
    }
    risk_clean = risk_raw.replace(risk_map)
    if 'Facility Type' in df_clean.columns:
        risk_mode_by_facility = risk_clean.groupby(df_clean['Facility Type']).transform(
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else pd.NA
        )
        risk_clean = risk_clean.fillna(risk_mode_by_facility)
    df_clean['Risk'] = risk_clean.fillna('Unknown')

if {'Inspection Date', 'LICENSE TERM START DATE'}.issubset(df_clean.columns):
    future_license_start_mask = (
        df_clean['Inspection Date'].notna()
        & df_clean['LICENSE TERM START DATE'].notna()
        & (df_clean['LICENSE TERM START DATE'] > df_clean['Inspection Date'])
    )
    df_clean.loc[future_license_start_mask, 'LICENSE TERM START DATE'] = pd.NaT
else:
    future_license_start_mask = pd.Series(False, index=df_clean.index)

if 'Violations' in df_clean.columns:
    violations_str = df_clean['Violations'].astype('string').str.strip()
    df_clean['violations_recorded'] = violations_str.notna() & violations_str.ne('')

if 'BL_LICENSE_ID' in df_clean.columns:
    df_clean['license_matched'] = df_clean['BL_LICENSE_ID'].notna()
elif 'LICENSE NUMBER' in df_clean.columns:
    df_clean['license_matched'] = df_clean['LICENSE NUMBER'].notna()
else:
    df_clean['license_matched'] = False

if 'Inspection Date' in df_clean.columns:
    prior_key = None
    for candidate in ['BL_LICENSE_ID', 'License #', 'LICENSE NUMBER', 'DBA Name', 'AKA Name']:
        if candidate in df_clean.columns:
            prior_key = candidate
            break
    if prior_key is not None:
        ordered_idx = df_clean.sort_values([prior_key, 'Inspection Date']).index
        has_prior = pd.Series(False, index=df_clean.index)
        has_prior.loc[ordered_idx] = (
            df_clean.loc[ordered_idx]
            .groupby(prior_key)
            .cumcount()
            .gt(0)
            .to_numpy()
        )
        df_clean['has_prior_inspection'] = has_prior
    else:
        df_clean['has_prior_inspection'] = False

if 'Location' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Location'])

log_step('cast_types_and_feature_safety', rows_before, df_clean.shape[0], cols_before, df_clean.shape[1], note='Type casting, Risk standardization, leakage fix, informative flags, drop Location')
print('Shape after type and value normalization:', df_clean.shape)
print('Rows with nulled future LICENSE TERM START DATE:', int(future_license_start_mask.sum()))

STAGE2_PATH.parent.mkdir(parents=True, exist_ok=True)
STAGE2_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(STAGE2_PATH, index=False)
save_log(STAGE2_LOG_PATH)
print('Saved stage 2 data to:', STAGE2_PATH)
print('Saved stage 2 log to:', STAGE2_LOG_PATH)

Shape after type and value normalization: (196825, 58)
Rows with nulled future LICENSE TERM START DATE: 145904
Saved stage 2 data to: ..\..\data\interim\cleaning\cleaning_stage2.csv
Saved stage 2 log to: ..\..\data\interim\cleaning\cleaning_stage2_log.csv
